# EVA02-Enormous LP-FT Training**Backbone:** `eva02_enormous_patch14_clip_224.laion2b_plus` (4.4B parameters)  **Method:** Linear Probing (5 epochs) → Fine-Tuning last 24/64 blocks with LLRD  **Input:** 336×336  **Seeds used across runs:** 42, 62, 82, 120  Change `CFG.seed` in the Configuration cell for each run.

## Imports & Configuration

In [ ]:
"""
EVA02-CLIP-Enormous (4.4B) - Resume-safe LP-FT
================================================
Stage 1: head-only linear probing
Stage 2: fine-tune last 24 blocks with LLRD

Fixes over prior version:
  - bf16 checkpoint compression (26 GB → ~10 GB)
  - mid-epoch saves are lightweight (no optimizer state)
  - label_smoothing 0.15 → 0.05 (teammate's winning recipe)
  - batch_size 16 → 32, grad_checkpointing OFF (speed on 102 GB GPU)
  - save frequency reduced to avoid disk thrashing
"""

import math
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

try:
	import timm, albumentations
except ImportError:
	import subprocess, sys
	subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'timm', 'albumentations'])

import gc
import json
import builtins as _builtins
import random
import shutil
import sys
import time
import warnings

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from contextlib import nullcontext
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

warnings.filterwarnings('ignore')


def first_existing(paths):
	for path in paths:
		if path and os.path.isdir(path):
			return path
	return None


def safe_torch_load(path, map_location='cpu'):
	"""Load checkpoint. Uses weights_only=False because our resume checkpoints
	contain numpy RNG state (from np.random.get_state()) which PyTorch 2.6+
	rejects under weights_only=True. All checkpoints are self-generated and trusted."""
	try:
		return torch.load(path, map_location=map_location, weights_only=False)
	except TypeError:
		# Ancient PyTorch without weights_only parameter
		return torch.load(path, map_location=map_location)


def unwrap_state_dict(obj):
	if not isinstance(obj, dict):
		return obj
	for key in ('state_dict', 'model_state_dict', 'model'):
		if key in obj and isinstance(obj[key], dict):
			return obj[key]
	return obj


def normalize_state_keys(state):
	out = {}
	for key, value in state.items():
		if key.startswith('module.'):
			key = key[7:]
		if key.startswith('backbone.'):
			key = key[len('backbone.'):]
		if key.startswith('model.'):
			key = key[len('model.'):]
		out[key] = value
	return out


def default_history():
	return {'tl': [], 'vl': [], 'tf': [], 'vf': [], 'ef': [], 'vp': [], 'lr': []}


def normalize_history(history):
	base = default_history()
	if not history:
		return base
	for key in base:
		if key in history:
			base[key] = history[key]
	return base


class CFG:
	seed = 42

	data_root = first_existing([
		'/kaggle/input/datasets/yashverdhan0/pig-classification-cropped/croppedpigdata/croppedpigdata',
	]) or '/kaggle/input/pig-posture-crops'
	crops_dir = os.path.join(data_root, 'crops')
	master_csv = os.path.join(data_root, 'master.csv')
	use_cache = True

	model_name = 'eva02_enormous_patch14_clip_224.laion2b_plus'
	pretrained_weights = '/kaggle/input/models/nulsable/evaenormous/pytorch/default/1/eva02_enormous_pretrained.pth'
	img_size = 336
	num_classes = 5

	# LP converges by epoch 3 (observed). Epochs 4-5 waste ~1h that FT needs.
	lp_epochs = 3
	lp_lr = 1e-3

	total_blocks = 64
	unfreeze_last_n = 24
	layer_decay = 0.80

	# --- BS=64 + grad ckpt: uses the spare 24GB VRAM for 2x smoother gradients ---
	# With class-balanced sampling, BS=64 gives ~13 samples/class/batch (vs ~6 at BS=32)
	# This makes the gradient signal for rare classes (Sit, LatL) much more stable
	# → the "breakthrough" moment becomes less random, more reliable
	# Gradient checkpointing trades ~30% speed/step for ~40GB VRAM savings
	# Net: epochs are FASTER (2x fewer steps, 1.3x slower per step = 0.65x epoch time)
	batch_size = 64
	num_workers = 10
	epochs = 50
	lr = 1.5e-5
	wd = 0.05
	head_wd = 0.01
	warmup_epochs_lp = 1
	# FT warmup=5: with BS=64 (half the steps/epoch vs BS=32), this gives ~3575 warmup steps
	# which matches the original recipe's 3 epochs × 1433 steps = 4299 steps
	warmup_epochs_ft = 5
	# --- FIXED: 0.05 per teammate's winning recipe ---
	label_smoothing = 0.05
	drop_path_rate = 0.35
	head_hidden_dim = 512
	head_dropout = 0.30
	head_mid_dropout = 0.20
	# ON: enables BS=64 within VRAM budget (78GB peak → ~55GB with ckpt + BS=64)
	gradient_checkpointing = True

	# Faster EMA: 0.9995 tracks current model more responsively than 0.9999
	# → catches the breakthrough moment instead of lagging behind it by ~10k batches
	ema_decay = 0.9995
	class2_weight = 3.0
	early_stopping_patience = 25  # patience for SWA to activate (epoch 25+)

	# --- SWA: start averaging weights from this epoch (0-indexed, must be >= lp_epochs) ---
	swa_start_epoch = 25

	time_limit_hours = 11.5
	time_limit_buffer_minutes = 30
	# --- FIXED: reduced save frequency to avoid disk thrashing ---
	save_every_batches = 500
	save_every_minutes = 30

	resume = True
	resume_from = ''
	output_prefix = 'eva02e_lpft'


IN_PAPERMILL = bool(os.environ.get('PAPERMILL_INPUT_PATH') or os.environ.get('PAPERMILL_OUTPUT_PATH'))
TRAIN_LOG_PATH = os.path.join(os.getcwd(), f'{CFG.output_prefix}_train.log') if IN_PAPERMILL else None
TRAIN_LOG_FILE = open(TRAIN_LOG_PATH, 'a', buffering=1) if TRAIN_LOG_PATH else None


def _should_echo_to_console(message):
	if not IN_PAPERMILL:
		return True
	important_prefixes = (
		'[DEVICE]',
		'=',
		'Train:',
		'Val:',
		'*** RESUMING',
		'[TIME LIMIT]',
		'[EARLY STOP]',
		'--- Stage 2:',
		'[Epoch ',
		'Ep ',
		'Done.',
		'Resume checkpoint:',
	)
	return message.startswith(important_prefixes)


def print(*args, **kwargs):
	message = kwargs.get('sep', ' ').join(str(arg) for arg in args)
	if TRAIN_LOG_FILE is not None:
		log_kwargs = dict(kwargs)
		log_kwargs['file'] = TRAIN_LOG_FILE
		_builtins.print(*args, **log_kwargs)
	if _should_echo_to_console(message):
		_builtins.print(*args, **kwargs)


if TRAIN_LOG_FILE is not None:
	import atexit
	atexit.register(TRAIN_LOG_FILE.close)


TEST_CAMERAS = ['pen1_tur_cam1', 'pen2_orb_cam2', 'pen2_tur_cam2']
CLASS_NAMES = {0: 'LatL', 1: 'LatR', 2: 'Sit', 3: 'Stand', 4: 'Stern'}


def seed_everything(seed):
	random.seed(seed)
	np.random.seed(seed)
	os.environ['PYTHONHASHSEED'] = str(seed)
	torch.manual_seed(seed)
	if torch.cuda.is_available():
		torch.cuda.manual_seed_all(seed)
		torch.backends.cudnn.benchmark = True
		torch.backends.cuda.matmul.allow_tf32 = True
		torch.backends.cudnn.allow_tf32 = True


seed_everything(CFG.seed)

WORKER_EPOCH_SEED = 0


def seed_worker(worker_id):
	seed = WORKER_EPOCH_SEED + worker_id
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	cv2.setRNGSeed(seed)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_TYPE = 'cuda' if torch.cuda.is_available() else 'cpu'
BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16

if torch.cuda.is_available():
	name = torch.cuda.get_device_name(0)
	total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
	print(f'[DEVICE] {name} ({total_gb:.0f} GB) | bf16: {BF16}', flush=True)


import torch.utils.checkpoint as _ckpt
if not getattr(_ckpt, '_patched_use_reentrant', False):
	_orig_checkpoint = _ckpt.checkpoint

	def _safe_checkpoint(fn, *args, **kwargs):
		kwargs.setdefault('use_reentrant', False)
		return _orig_checkpoint(fn, *args, **kwargs)

	_ckpt.checkpoint = _safe_checkpoint
	_ckpt._patched_use_reentrant = True


def autocast_context():
	if DEVICE_TYPE == 'cuda':
		return torch.amp.autocast('cuda', dtype=AMP_DTYPE)
	return nullcontext()


## Checkpoint CompressionCompress tensors to bf16 to cut checkpoint size from 26 GB → ~10 GB.

In [ ]:
def _compress_tensor(t):
	"""Compress a single tensor to bf16 if it's a large float tensor."""
	if isinstance(t, torch.Tensor) and t.is_floating_point() and t.numel() > 1:
		return t.to(torch.bfloat16)
	return t


def compress_state_dict(state):
	"""Compress all large float tensors in a state dict to bf16."""
	return {k: _compress_tensor(v) for k, v in state.items()}


def compress_optimizer_state(opt_state):
	"""Compress optimizer momentum/variance buffers to bf16.
	Preserves step counters and param_groups metadata exactly."""
	if opt_state is None:
		return None
	result = dict(opt_state)
	if 'state' in result:
		compressed = {}
		for param_id, param_state in result['state'].items():
			compressed[param_id] = {
				k: _compress_tensor(v) if k in ('exp_avg', 'exp_avg_sq') else v
				for k, v in param_state.items()
			}
		result['state'] = compressed
	return result


## Layer-wise Learning Rate Decay (LLRD)Each backbone block gets its own LR: `lr_block_i = base_lr * decay^(top_block - i)`

In [ ]:
def get_block_id(name, total_blocks):
	if name.startswith('head.'):
		return total_blocks + 1
	if 'patch_embed' in name or 'pos_embed' in name or 'cls_token' in name:
		return 0
	if 'blocks.' in name:
		return int(name.split('blocks.')[1].split('.')[0]) + 1
	return total_blocks + 1


def make_stage1_groups(model):
	decay, no_decay = [], []
	for name, param in model.head.named_parameters():
		if not param.requires_grad:
			continue
		if param.ndim == 1 or name.endswith('bias') or 'norm' in name or 'ln' in name:
			no_decay.append(param)
		else:
			decay.append(param)
	groups = []
	if decay:
		groups.append({'params': decay, 'lr': CFG.lp_lr, 'weight_decay': CFG.head_wd})
	if no_decay:
		groups.append({'params': no_decay, 'lr': CFG.lp_lr, 'weight_decay': 0.0})
	return groups


def make_stage2_groups(model):
	groups = {}
	for name, param in model.named_parameters():
		if not param.requires_grad:
			continue
		layer_id = get_block_id(name, CFG.total_blocks)
		is_nodecay = param.ndim == 1 or name.endswith('bias') or 'norm' in name or 'ln' in name
		wd = 0.0 if is_nodecay else (CFG.head_wd if name.startswith('head.') else CFG.wd)
		if layer_id <= CFG.total_blocks:
			depth = CFG.total_blocks + 1 - layer_id
			lr = CFG.lr * (CFG.layer_decay ** depth)
		else:
			lr = CFG.lr
		key = (layer_id, wd)
		if key not in groups:
			groups[key] = {'params': [], 'lr': lr, 'weight_decay': wd, '_layer_id': layer_id}
		groups[key]['params'].append(param)
	ordered = sorted(groups.values(), key=lambda g: (g['_layer_id'], g['weight_decay']))
	for group in ordered:
		group.pop('_layer_id', None)
	return ordered


## Exponential Moving Average (EMA)CPU-backed EMA to avoid doubling GPU memory.

In [ ]:
class CPUBackedEMA:
	def __init__(self, model, decay=0.9999):
		self.decay = decay
		self.shadow = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
		self.backup = {}

	def extend(self, model):
		for name, param in model.named_parameters():
			if param.requires_grad and name not in self.shadow:
				self.shadow[name] = param.detach().cpu().clone()

	@torch.no_grad()
	def update(self, model):
		for name, param in model.named_parameters():
			if param.requires_grad and name in self.shadow:
				self.shadow[name].mul_(self.decay).add_(param.detach().cpu(), alpha=1.0 - self.decay)

	def apply(self, model):
		self.backup = {}
		for name, param in model.named_parameters():
			if param.requires_grad and name in self.shadow:
				self.backup[name] = param.detach().cpu().clone()
				param.data.copy_(self.shadow[name].to(device=param.device, dtype=param.dtype))

	def restore(self, model):
		for name, param in model.named_parameters():
			if param.requires_grad and name in self.backup:
				param.data.copy_(self.backup[name].to(device=param.device, dtype=param.dtype))
		self.backup = {}

	def state_dict(self):
		return {name: tensor.clone() for name, tensor in self.shadow.items()}

	def load_state_dict(self, state):
		self.shadow = {name: tensor.clone() for name, tensor in state.items()}


## SWA (Stochastic Weight Averaging)

In [ ]:
class CPUBackedSWA:
	"""Uniform running average of trainable weights, stored on CPU.
	Unlike EMA (exponential decay → favors recent weights), SWA gives EQUAL
	weight to every snapshot → captures the center of all basins visited
	during training → flatter minimum → better generalization.

	No disk I/O: the average lives in CPU RAM and is saved once at end of training.
	"""

	def __init__(self):
		self.shadow = {}   # running average on CPU
		self.n = 0         # number of snapshots accumulated

	def is_active(self):
		return self.n > 0

	@torch.no_grad()
	def update(self, model):
		"""Accumulate one snapshot using the numerically-stable running average:
		avg_{n} = avg_{n-1} + (x_n - avg_{n-1}) / n"""
		self.n += 1
		for name, param in model.named_parameters():
			if not param.requires_grad:
				continue
			p_cpu = param.detach().cpu()
			if name not in self.shadow:
				self.shadow[name] = p_cpu.clone()
			else:
				self.shadow[name].add_((p_cpu - self.shadow[name]) / self.n)

	def state_dict(self):
		return {name: tensor.clone() for name, tensor in self.shadow.items()}

	def load_state_dict(self, state, n):
		self.shadow = {name: tensor.clone() for name, tensor in state.items()}
		self.n = n


## Data AugmentationsTrain: ColorJitter, GaussianBlur, GaussNoise, Downscale, CLAHE.Val: Resize + Normalize only.

In [ ]:
def get_train_transforms():
	# Exact proven recipe from 0.858 LB run — do not add to this
	return A.Compose([
		A.Resize(CFG.img_size, CFG.img_size),
		A.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.05, p=0.50),
		A.GaussianBlur(blur_limit=(3, 5), p=0.20),
		A.GaussNoise(p=0.20),
		A.Downscale(scale_min=0.70, scale_max=0.95, p=0.15),
		A.CLAHE(clip_limit=2.0, p=0.20),
		A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
		ToTensorV2(),
	])


def get_val_transforms():
	return A.Compose([
		A.Resize(CFG.img_size, CFG.img_size),
		A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
		ToTensorV2(),
	])


## DatasetLabel-aware horizontal flip: swaps LatL ↔ LatR when flipping.Optional RAM caching for speed.

In [ ]:
class PigDataset(Dataset):
	def __init__(self, df, crops_dir, transforms=None, is_train=True, has_label=True, cache=False):
		self.df = df.reset_index(drop=True)
		self.transforms = transforms
		self.is_train = is_train
		self.has_label = has_label
		self.crops_dir = crops_dir
		self.cached_bytes = None

		if cache:
			print(f'  Caching {len(self.df)} images into RAM...', end=' ', flush=True)
			t0 = time.time()
			self.cached_bytes = []
			for crop_name in self.df['crop_filename']:
				with open(os.path.join(self.crops_dir, crop_name), 'rb') as f:
					self.cached_bytes.append(f.read())
			size_mb = sum(len(b) for b in self.cached_bytes) / 1e6
			print(f'{size_mb:.0f} MB in {time.time() - t0:.1f}s', flush=True)

	def __len__(self):
		return len(self.df)

	def __getitem__(self, idx):
		if self.cached_bytes is not None:
			buf = np.frombuffer(self.cached_bytes[idx], dtype=np.uint8)
			image = cv2.imdecode(buf, cv2.IMREAD_COLOR)
		else:
			image = cv2.imread(os.path.join(self.crops_dir, self.df.iloc[idx]['crop_filename']), cv2.IMREAD_COLOR)

		image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

		if self.has_label:
			label = int(self.df.iloc[idx]['class_id'])
			if self.is_train and random.random() < 0.5:
				image = cv2.flip(image, 1)
				if label == 0:
					label = 1
				elif label == 1:
					label = 0
			if self.transforms:
				image = self.transforms(image=image)['image']
			return image, torch.tensor(label, dtype=torch.long)

		if self.transforms:
			image = self.transforms(image=image)['image']
		return image, self.df.iloc[idx]['row_id']


## Model ArchitectureEVA02-Enormous backbone (4.4B) + MLP head:`LayerNorm → Dropout → Linear(1792→512) → GELU → Dropout → Linear(512→5)`

In [ ]:
def interpolate_pos_embed(state, backbone):
	if 'pos_embed' not in state or not hasattr(backbone, 'pos_embed'):
		return state
	old_pe = state['pos_embed']
	new_pe = backbone.pos_embed
	if old_pe.shape == new_pe.shape:
		return state
	dim = old_pe.shape[-1]
	cls_tok = old_pe[:, :1, :]
	patch_pe = old_pe[:, 1:, :]
	old_grid = int(patch_pe.shape[1] ** 0.5)
	new_grid = int((new_pe.shape[1] - 1) ** 0.5)
	patch_pe = patch_pe.reshape(1, old_grid, old_grid, dim).permute(0, 3, 1, 2).float()
	patch_pe = F.interpolate(patch_pe, size=(new_grid, new_grid), mode='bicubic', align_corners=False)
	patch_pe = patch_pe.permute(0, 2, 3, 1).reshape(1, new_grid * new_grid, dim)
	state['pos_embed'] = torch.cat([cls_tok, patch_pe], dim=1)
	return state


class PigModel(nn.Module):
	def __init__(self):
		super().__init__()
		self.backbone = timm.create_model(
			CFG.model_name,
			pretrained=False,
			num_classes=0,
			img_size=CFG.img_size,
			drop_path_rate=CFG.drop_path_rate,
		)

		if CFG.pretrained_weights and os.path.exists(CFG.pretrained_weights):
			print(f'  Loading pretrained weights from: {CFG.pretrained_weights}', flush=True)
			state = safe_torch_load(CFG.pretrained_weights, map_location='cpu')
			state = normalize_state_keys(unwrap_state_dict(state))
			state = {k: v for k, v in state.items() if not k.startswith('head.') and not k.startswith('fc_norm.')}
			state = interpolate_pos_embed(state, self.backbone)
			msg = self.backbone.load_state_dict(state, strict=False)
			print(f'  Loaded {len(state)} keys | missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}', flush=True)
			del state
			gc.collect()
		else:
			print(f'  WARNING: pretrained weights not found at {CFG.pretrained_weights}', flush=True)

		if CFG.gradient_checkpointing and hasattr(self.backbone, 'set_grad_checkpointing'):
			self.backbone.set_grad_checkpointing(True)
			print('  Gradient checkpointing: ON', flush=True)

		feat_dim = self.backbone.num_features
		# Main 5-class head
		self.head = nn.Sequential(
			nn.LayerNorm(feat_dim),
			nn.Dropout(CFG.head_dropout),
			nn.Linear(feat_dim, CFG.head_hidden_dim),
			nn.GELU(),
			nn.Dropout(CFG.head_mid_dropout),
			nn.Linear(CFG.head_hidden_dim, CFG.num_classes),
		)
		nn.init.normal_(self.head[-1].weight, std=0.01)
		nn.init.zeros_(self.head[-1].bias)

	def forward(self, x):
		features = self.backbone(x)
		return self.head(features)


class ClassWeightedCE(nn.Module):
	def __init__(self):
		super().__init__()
		weights = torch.ones(CFG.num_classes, dtype=torch.float32)
		weights[2] = CFG.class2_weight
		weights /= weights.mean()
		self.register_buffer('weights', weights)

	def forward(self, logits, targets):
		return F.cross_entropy(logits, targets, weight=self.weights, label_smoothing=CFG.label_smoothing)


## RNG State Capture/RestoreFor exact reproducibility across resume checkpoints.

In [ ]:
def capture_rng_state():
	state = {
		'python': random.getstate(),
		'numpy': np.random.get_state(),
		'torch': torch.get_rng_state(),
	}
	if torch.cuda.is_available():
		state['cuda'] = torch.cuda.get_rng_state_all()
	return state


def restore_rng_state(state):
	if not state:
		return
	random.setstate(state['python'])
	np.random.set_state(state['numpy'])
	torch.set_rng_state(state['torch'])
	if torch.cuda.is_available() and 'cuda' in state:
		torch.cuda.set_rng_state_all(state['cuda'])


## Trainable State HelpersExtract/load only the trainable parameters (not the frozen backbone).

In [ ]:
def extract_trainable_state(model):
	return {name: param.detach().cpu().clone() for name, param in model.named_parameters() if param.requires_grad}


def load_trainable_state(model, state):
	loaded = 0
	for name, param in model.named_parameters():
		if param.requires_grad and name in state:
			param.data.copy_(state[name].to(device=param.device, dtype=param.dtype))
			loaded += 1
	return loaded


## Checkpoint Save / LoadResume-safe checkpointing with mid-epoch saves.Lightweight saves skip optimizer state to avoid disk overflow.

In [ ]:
def save_checkpoint(path, payload):
	tmp_path = f'{path}.tmp'
	if os.path.exists(path):
		os.remove(path)
	if os.path.exists(tmp_path):
		os.remove(tmp_path)
	torch.save(payload, tmp_path)
	os.replace(tmp_path, path)
	size_mb = os.path.getsize(path) / 1e6
	free_gb = shutil.disk_usage(os.path.dirname(path) or '.').free / 1e9
	print(f'  Saved {os.path.basename(path)}: {size_mb:.0f} MB | free={free_gb:.1f} GB', flush=True)


def build_resume_payload(model, optimizer, scheduler, ema, swa, scaler, epoch, batch_idx, stage,
						 epoch_seed, best_f1, best_ep, no_imp, history, train_accum,
						 lightweight=False):
	"""Build checkpoint payload.
	lightweight=True: skip optimizer state (for mid-epoch saves to save disk).
	All float tensors are compressed to bf16 to cut checkpoint size in half.
	"""
	payload = {
		'meta': {
			'schema': 'eva02e_lpft_resume_v2',
			'model_name': CFG.model_name,
			'img_size': CFG.img_size,
			'head_type': 'mlp',
			'head_hidden_dim': CFG.head_hidden_dim,
			'unfreeze_last_n': CFG.unfreeze_last_n,
			'layer_decay': CFG.layer_decay,
			'lp_epochs': CFG.lp_epochs,
			'num_classes': CFG.num_classes,
			'lightweight': lightweight,
		},
		'epoch': epoch,
		'batch_idx': batch_idx,
		'stage': stage,
		'epoch_seed': epoch_seed,
		'best_f1': best_f1,
		'best_ep': best_ep,
		'no_imp': no_imp,
		'history': history,
		'train_accum': train_accum,
		'trainable_state': compress_state_dict(extract_trainable_state(model)),
		'ema_state': compress_state_dict(ema.state_dict()),
		'rng_state': capture_rng_state(),
	}
	# Only include optimizer/scheduler/scaler/SWA state in full (non-lightweight) saves
	if not lightweight:
		payload['optimizer_state'] = compress_optimizer_state(optimizer.state_dict())
		payload['scheduler_state'] = scheduler.state_dict() if scheduler is not None else None
		payload['scaler_state'] = scaler.state_dict() if scaler is not None and scaler.is_enabled() else None
		# SWA state (only in full saves — it's large but essential for resume)
		if swa.is_active():
			payload['swa_state'] = compress_state_dict(swa.state_dict())
			payload['swa_n'] = swa.n
	else:
		payload['optimizer_state'] = None
		payload['scheduler_state'] = None
		payload['scaler_state'] = None
	return payload


def build_best_payload(model, ema, use_ema, stage, best_f1, best_ep):
	return {
		'meta': {
			'schema': 'eva02e_lpft_best_v2',
			'model_name': CFG.model_name,
			'img_size': CFG.img_size,
			'head_type': 'mlp',
			'head_hidden_dim': CFG.head_hidden_dim,
			'unfreeze_last_n': CFG.unfreeze_last_n,
			'layer_decay': CFG.layer_decay,
			'num_classes': CFG.num_classes,
		},
		'stage': stage,
		'use_ema': use_ema,
		'best_f1': best_f1,
		'best_ep': best_ep,
		'trainable_state': compress_state_dict(
			ema.state_dict() if use_ema else extract_trainable_state(model)
		),
	}


def build_swa_payload(swa, stage):
	"""Build a lightweight payload for the SWA model — just weights + metadata."""
	return {
		'meta': {
			'schema': 'eva02e_lpft_swa_v1',
			'model_name': CFG.model_name,
			'img_size': CFG.img_size,
			'head_type': 'mlp',
			'head_hidden_dim': CFG.head_hidden_dim,
			'unfreeze_last_n': CFG.unfreeze_last_n,
			'layer_decay': CFG.layer_decay,
			'num_classes': CFG.num_classes,
			'swa_snapshots': swa.n,
		},
		'stage': stage,
		'use_ema': False,
		'best_f1': 0.0,  # SWA F1 is evaluated post-hoc at inference
		'best_ep': 0,
		'trainable_state': compress_state_dict(swa.state_dict()),
	}


def save_resume_checkpoint(path, model, optimizer, scheduler, ema, swa, scaler, epoch, batch_idx, stage,
						   epoch_seed, best_f1, best_ep, no_imp, history, train_accum,
						   lightweight=False):
	payload = build_resume_payload(
		model, optimizer, scheduler, ema, swa, scaler, epoch, batch_idx, stage,
		epoch_seed, best_f1, best_ep, no_imp, history, train_accum,
		lightweight=lightweight,
	)
	save_checkpoint(path, payload)


def save_best_checkpoint(path, model, ema, use_ema, stage, best_f1, best_ep):
	save_checkpoint(path, build_best_payload(model, ema, use_ema, stage, best_f1, best_ep))


def load_resume_state(path):
	if not path or not os.path.exists(path):
		return None
	ckpt = safe_torch_load(path, map_location='cpu')
	if not isinstance(ckpt, dict):
		return None
	return {
		'epoch': int(ckpt.get('epoch', 0)),
		'batch_idx': int(ckpt.get('batch_idx', 0)),
		'stage': int(ckpt.get('stage', 1 if ckpt.get('epoch', 0) < CFG.lp_epochs else 2)),
		'epoch_seed': int(ckpt.get('epoch_seed', CFG.seed)),
		'best_f1': float(ckpt.get('best_f1', 0.0)),
		'best_ep': int(ckpt.get('best_ep', 0)),
		'no_imp': int(ckpt.get('no_imp', 0)),
		'history': normalize_history(ckpt.get('history', ckpt.get('hist'))),
		'train_accum': ckpt.get('train_accum', {
			'loss_sum': 0.0, 'seen': 0, 'preds': [], 'labels': [],
		}),
		'trainable_state': ckpt.get('trainable_state', ckpt.get('trainable', {})),
		'optimizer_state': ckpt.get('optimizer_state', ckpt.get('opt_state')),
		'scheduler_state': ckpt.get('scheduler_state', ckpt.get('sched_state')),
		'ema_state': ckpt.get('ema_state', ckpt.get('ema', {})),
		'scaler_state': ckpt.get('scaler_state'),
		'rng_state': ckpt.get('rng_state'),
		'meta': ckpt.get('meta', {}),
	}


## Optimizer & Scheduler- Stage 1 (LP): AdamW, head-only, LR=1e-3- Stage 2 (FT): AdamW with LLRD, LR=1.5e-5, cosine schedule

In [ ]:
def build_stage1_optimizer(model):
	return torch.optim.AdamW(make_stage1_groups(model), betas=(0.9, 0.999), eps=1e-8)


def build_stage2_optimizer(model):
	return torch.optim.AdamW(make_stage2_groups(model), betas=(0.9, 0.999), eps=1e-8)


def build_stage1_scheduler(optimizer):
	if CFG.lp_epochs <= CFG.warmup_epochs_lp:
		return CosineAnnealingLR(optimizer, T_max=max(1, CFG.lp_epochs), eta_min=1e-5)
	warmup = LinearLR(optimizer, start_factor=0.1, total_iters=CFG.warmup_epochs_lp)
	cosine = CosineAnnealingLR(optimizer, T_max=max(1, CFG.lp_epochs - CFG.warmup_epochs_lp), eta_min=1e-5)
	return SequentialLR(optimizer, [warmup, cosine], milestones=[CFG.warmup_epochs_lp])


def build_stage2_scheduler(optimizer):
	total_ft_epochs = max(1, CFG.epochs - CFG.lp_epochs)
	if total_ft_epochs <= CFG.warmup_epochs_ft:
		return CosineAnnealingLR(optimizer, T_max=total_ft_epochs, eta_min=1e-7)
	warmup = LinearLR(optimizer, start_factor=0.1, total_iters=CFG.warmup_epochs_ft)
	cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_ft_epochs - CFG.warmup_epochs_ft), eta_min=1e-7)
	return SequentialLR(optimizer, [warmup, cosine], milestones=[CFG.warmup_epochs_ft])


## Trainability Control- Stage 1: freeze everything except head- Stage 2: unfreeze last 24/64 blocks + final norm

In [ ]:
def set_stage1_trainability(model):
	for param in model.parameters():
		param.requires_grad = False
	for param in model.head.parameters():
		param.requires_grad = True


def set_stage2_trainability(model):
	for param in model.parameters():
		param.requires_grad = False
	for param in model.head.parameters():
		param.requires_grad = True
	total = len(model.backbone.blocks)
	start = max(0, total - CFG.unfreeze_last_n)
	for idx in range(start, total):
		for param in model.backbone.blocks[idx].parameters():
			param.requires_grad = True
	for attr in ('norm', 'fc_norm'):
		if hasattr(model.backbone, attr):
			for param in getattr(model.backbone, attr).parameters():
				param.requires_grad = True


## Data LoadersClass-balanced sampling via WeightedRandomSampler.Re-seeded every epoch for reproducibility.

In [ ]:
def make_train_loader(dataset, df, epoch_seed):
	# Class-only balanced sampling (proven recipe — no camera reweighting)
	class_counts = df['class_id'].value_counts().sort_index().reindex(range(CFG.num_classes), fill_value=1).values
	sample_weights = [1.0 / class_counts[int(label)] for label in df['class_id'].values]
	generator = torch.Generator()
	generator.manual_seed(epoch_seed)
	global WORKER_EPOCH_SEED
	WORKER_EPOCH_SEED = epoch_seed
	try:
		sampler = WeightedRandomSampler(
			weights=torch.as_tensor(sample_weights, dtype=torch.double),
			num_samples=len(sample_weights),
			replacement=True,
			generator=generator,
		)
	except TypeError:
		sampler = WeightedRandomSampler(
			weights=torch.as_tensor(sample_weights, dtype=torch.double),
			num_samples=len(sample_weights),
			replacement=True,
		)
	return DataLoader(
		dataset,
		batch_size=CFG.batch_size,
		sampler=sampler,
		num_workers=CFG.num_workers,
		pin_memory=(DEVICE_TYPE == 'cuda'),
		drop_last=True,
		worker_init_fn=seed_worker,
		persistent_workers=False,
	)


def make_eval_loader(dataset):
	return DataLoader(
		dataset,
		batch_size=CFG.batch_size,
		shuffle=False,
		num_workers=CFG.num_workers,
		pin_memory=(DEVICE_TYPE == 'cuda'),
		persistent_workers=False,
	)


## EvaluationCompute both raw and EMA predictions, pick whichever has higher macro-F1.

In [ ]:
def evaluate(model, ema, loader, criterion):
	model.eval()
	raw_loss = 0.0
	seen = 0
	raw_preds = []
	labels = []

	with torch.no_grad():
		for imgs, lbls in loader:
			imgs = imgs.to(DEVICE, non_blocking=True)
			lbls = lbls.to(DEVICE, non_blocking=True)
			with autocast_context():
				main_logits = model(imgs)
				loss = criterion(main_logits, lbls)
			raw_loss += loss.item() * imgs.size(0)
			seen += imgs.size(0)
			raw_preds.extend(main_logits.argmax(1).detach().cpu().tolist())
			labels.extend(lbls.detach().cpu().tolist())

	ema.apply(model)
	ema_preds = []
	try:
		with torch.no_grad():
			for imgs, _ in loader:
				imgs = imgs.to(DEVICE, non_blocking=True)
				with autocast_context():
					main_logits = model(imgs)
				ema_preds.extend(main_logits.argmax(1).detach().cpu().tolist())
	finally:
		ema.restore(model)

	labels = np.asarray(labels)
	raw_preds = np.asarray(raw_preds)
	ema_preds = np.asarray(ema_preds)

	raw_f1 = f1_score(labels, raw_preds, average='macro')
	ema_f1 = f1_score(labels, ema_preds, average='macro')
	raw_pc = f1_score(labels, raw_preds, average=None, labels=list(range(CFG.num_classes)))
	ema_pc = f1_score(labels, ema_preds, average=None, labels=list(range(CFG.num_classes)))
	use_ema = ema_f1 >= raw_f1

	return {
		'val_loss': raw_loss / max(1, seen),
		'raw_f1': float(raw_f1),
		'ema_f1': float(ema_f1),
		'best_f1': float(ema_f1 if use_ema else raw_f1),
		'best_pc': [float(x) for x in (ema_pc if use_ema else raw_pc)],
		'use_ema': use_ema,
	}


## Convergence DashboardSave loss curves, F1 curves, per-class F1, and LR schedule as PNG.

In [ ]:
def save_dashboard(history, best_f1, best_ep, work_dir, prefix):
	"""Save convergence graphs as PNG. Called before any exit."""
	if not history.get('tl') or len(history['tl']) < 2:
		return
	try:
		epochs_x = list(range(1, len(history['tl']) + 1))
		fig, axes = plt.subplots(2, 2, figsize=(16, 12))
		fig.suptitle(f'{prefix} | Best F1={best_f1:.4f} @ Ep {best_ep}', fontsize=14, fontweight='bold')

		# --- Panel 1: Loss ---
		ax = axes[0, 0]
		ax.plot(epochs_x, history['tl'], 'b-', label='Train Loss', linewidth=1.5)
		ax.plot(epochs_x, history['vl'], 'r-', label='Val Loss', linewidth=1.5)
		ax.set_xlabel('Epoch')
		ax.set_ylabel('Loss')
		ax.set_title('Loss Curves')
		ax.legend()
		ax.grid(True, alpha=0.3)

		# --- Panel 2: Macro F1 ---
		ax = axes[0, 1]
		ax.plot(epochs_x, history['tf'], 'b--', label='Train F1', linewidth=1.5)
		ax.plot(epochs_x, history['vf'], 'r-', label='Val F1 (raw)', linewidth=1.5)
		if history.get('ef'):
			ax.plot(epochs_x[:len(history['ef'])], history['ef'], 'g-', label='Val F1 (EMA)', linewidth=1.5)
		if best_ep > 0:
			ax.axvline(x=best_ep, color='gold', linestyle=':', linewidth=2, label=f'Best @ Ep {best_ep}')
		ax.set_xlabel('Epoch')
		ax.set_ylabel('Macro F1')
		ax.set_title('F1 Convergence')
		ax.legend()
		ax.grid(True, alpha=0.3)

		# --- Panel 3: Per-class F1 ---
		ax = axes[1, 0]
		if history.get('vp') and len(history['vp']) > 0:
			n_cls = len(history['vp'][0])
			for c in range(n_cls):
				vals = [ep_pc[c] for ep_pc in history['vp'] if len(ep_pc) > c]
				ax.plot(epochs_x[:len(vals)], vals, '-', label=CLASS_NAMES.get(c, f'C{c}'), linewidth=1.5)
		ax.set_xlabel('Epoch')
		ax.set_ylabel('F1')
		ax.set_title('Per-Class F1 (Val)')
		ax.legend()
		ax.grid(True, alpha=0.3)

		# --- Panel 4: Learning Rate ---
		ax = axes[1, 1]
		if history.get('lr'):
			ax.plot(epochs_x[:len(history['lr'])], history['lr'], 'k-', linewidth=1.5)
		ax.set_xlabel('Epoch')
		ax.set_ylabel('LR')
		ax.set_title('Learning Rate Schedule')
		ax.set_yscale('log')
		ax.grid(True, alpha=0.3)

		plt.tight_layout()
		path = os.path.join(work_dir, f'{prefix}_dashboard.png')
		fig.savefig(path, dpi=120, bbox_inches='tight')
		plt.close(fig)
		print(f'  Dashboard saved: {path}', flush=True)
	except Exception as e:
		print(f'  WARNING: dashboard save failed: {e}', flush=True)


## Training Loop (One Epoch)11.5-hour time limit with clean mid-epoch checkpoint saves.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, ema, swa, scaler, scheduler, epoch, stage,
					epoch_seed, start_batch, deadline, resume_path, history, best_f1, best_ep,
					no_imp, train_accum):
	model.train()
	trainable_params = [p for p in model.parameters() if p.requires_grad]
	loader_len = len(loader)
	start_batch = min(start_batch, loader_len)
	loader_iter = iter(loader)
	for _ in range(start_batch):
		next(loader_iter, None)

	running_loss = float(train_accum.get('loss_sum', 0.0))
	seen = int(train_accum.get('seen', 0))
	preds = list(train_accum.get('preds', []))
	labels = list(train_accum.get('labels', []))
	last_save = time.time()
	last_log = time.time()

	for batch_idx in range(start_batch, loader_len):
		imgs, lbls = next(loader_iter)
		imgs = imgs.to(DEVICE, non_blocking=True)
		lbls = lbls.to(DEVICE, non_blocking=True)

		optimizer.zero_grad(set_to_none=True)
		with autocast_context():
			main_logits = model(imgs)
			loss = criterion(main_logits, lbls)

		if scaler.is_enabled():
			scaler.scale(loss).backward()
			scaler.unscale_(optimizer)
			torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
			scaler.step(optimizer)
			scaler.update()
		else:
			loss.backward()
			torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
			optimizer.step()

		ema.update(model)

		batch_size = imgs.size(0)
		running_loss += loss.item() * batch_size
		seen += batch_size
		preds.extend(main_logits.argmax(1).detach().cpu().tolist())
		labels.extend(lbls.detach().cpu().tolist())

		next_batch = batch_idx + 1
		now = time.time()
		if next_batch == 1 or (now - last_log) >= 60 or next_batch == loader_len:
			stg = 'LP' if stage == 1 else 'FT'
			pct = 100.0 * next_batch / loader_len
			print(f'  [{stg}] ep{epoch + 1} {next_batch}/{loader_len} ({pct:5.1f}%) | loss={loss.item():.4f}', flush=True)
			last_log = now

		# Mid-epoch saves are LIGHTWEIGHT (no optimizer state) to avoid disk overflow
		should_save = (
			(CFG.save_every_batches > 0 and next_batch % CFG.save_every_batches == 0)
			or ((now - last_save) >= CFG.save_every_minutes * 60)
		)
		if should_save:
			save_resume_checkpoint(
				resume_path, model, optimizer, scheduler, ema, swa, scaler,
				epoch, next_batch, stage, epoch_seed, best_f1, best_ep, no_imp,
				history, {'loss_sum': running_loss, 'seen': seen, 'preds': preds, 'labels': labels},
				lightweight=True,
			)
			last_save = time.time()

		if time.time() > deadline:
			print('[TIME LIMIT] checkpoint saved mid-epoch; stopping cleanly.', flush=True)
			# Time limit save: try full save, fall back to lightweight
			try:
				save_resume_checkpoint(
					resume_path, model, optimizer, scheduler, ema, swa, scaler,
					epoch, next_batch, stage, epoch_seed, best_f1, best_ep, no_imp,
					history, {'loss_sum': running_loss, 'seen': seen, 'preds': preds, 'labels': labels},
					lightweight=False,
				)
			except OSError:
				print('  Full save failed (disk full), saving lightweight...', flush=True)
				save_resume_checkpoint(
					resume_path, model, optimizer, scheduler, ema, swa, scaler,
					epoch, next_batch, stage, epoch_seed, best_f1, best_ep, no_imp,
					history, {'loss_sum': running_loss, 'seen': seen, 'preds': preds, 'labels': labels},
					lightweight=True,
				)
			return {
				'interrupted': True,
				'train_loss': running_loss / max(1, seen),
				'train_f1': f1_score(labels, preds, average='macro') if labels else 0.0,
				'next_batch': next_batch,
				'train_accum': {'loss_sum': running_loss, 'seen': seen, 'preds': preds, 'labels': labels},
			}

	return {
		'interrupted': False,
		'train_loss': running_loss / max(1, seen),
		'train_f1': f1_score(labels, preds, average='macro') if labels else 0.0,
		'next_batch': 0,
		'train_accum': {'loss_sum': running_loss, 'seen': seen, 'preds': preds, 'labels': labels},
	}


## Main Entry PointFull LP-FT pipeline: Stage 1 (head-only) → Stage 2 (partial unfreeze).Resume-safe: picks up exactly where it left off.

In [ ]:
def main():
	gc.collect()
	if torch.cuda.is_available():
		torch.cuda.empty_cache()
		torch.cuda.reset_peak_memory_stats()

	global_start = time.time()
	deadline = global_start + CFG.time_limit_hours * 3600 - CFG.time_limit_buffer_minutes * 60

	work_dir = os.getcwd()
	resume_path = os.path.join(work_dir, f'{CFG.output_prefix}_resume.pth')
	best_path = os.path.join(work_dir, f'{CFG.output_prefix}_best.pth')
	history_path = os.path.join(work_dir, f'{CFG.output_prefix}_history.json')

	if CFG.resume_from and os.path.exists(CFG.resume_from) and not os.path.exists(resume_path):
		shutil.copy2(CFG.resume_from, resume_path)
		print(f'  Copied resume checkpoint from {CFG.resume_from}', flush=True)

	# Also copy best.pth so it exists in working dir even if model never beats previous best
	_best_from = CFG.resume_from.replace('_resume.pth', '_best.pth')
	if os.path.exists(_best_from) and not os.path.exists(best_path):
		shutil.copy2(_best_from, best_path)
		print(f'  Copied best checkpoint from {_best_from}', flush=True)

	print('=' * 80)
	print(f'  EVA02-Enormous LP-FT | {CFG.model_name}')
	print(f'  data_root: {CFG.data_root}')
	print(f'  LP epochs: {CFG.lp_epochs} | unfreeze last {CFG.unfreeze_last_n}/{CFG.total_blocks} blocks')
	print(f'  img_size: {CFG.img_size} | lp_lr: {CFG.lp_lr:.1e} | ft_lr: {CFG.lr:.1e} | layer_decay: {CFG.layer_decay}')
	print(f'  label_smoothing: {CFG.label_smoothing} | drop_path: {CFG.drop_path_rate} | head_hidden: {CFG.head_hidden_dim}')
	print(f'  batch_size: {CFG.batch_size} | grad_ckpt: {CFG.gradient_checkpointing} | class2_wt: {CFG.class2_weight}')
	print(f'  time_limit: {CFG.time_limit_hours:.1f}h | buffer: {CFG.time_limit_buffer_minutes}m')
	print('=' * 80, flush=True)

	df = pd.read_csv(CFG.master_csv)
	df_tp = df[df['set'].isin(['train1', 'train2'])].copy().reset_index(drop=True)
	df_tp['cam_group'] = df_tp['pen'].astype(str) + '_' + df_tp['cam_type'].astype(str) + '_' + df_tp['cam_num'].astype(str)
	val_mask = df_tp['cam_group'].isin(TEST_CAMERAS)
	df_tr = df_tp[~val_mask].reset_index(drop=True)
	df_va = df_tp[val_mask].reset_index(drop=True)

	print(f'\nTrain: {len(df_tr)} | Val: {len(df_va)}')
	print(f'Train dist: {df_tr["class_id"].value_counts().sort_index().to_dict()}')
	print(f'Val dist:   {df_va["class_id"].value_counts().sort_index().to_dict()}')

	tr_ds = PigDataset(df_tr, CFG.crops_dir, get_train_transforms(), is_train=True, has_label=True, cache=CFG.use_cache)
	va_ds = PigDataset(df_va, CFG.crops_dir, get_val_transforms(), is_train=False, has_label=True, cache=CFG.use_cache)
	va_loader = make_eval_loader(va_ds)

	model = PigModel().to(DEVICE)
	CFG.total_blocks = len(model.backbone.blocks)
	print(f'  Backbone blocks: {CFG.total_blocks}', flush=True)

	criterion = ClassWeightedCE().to(DEVICE)

	resume_state = load_resume_state(resume_path) if CFG.resume and os.path.exists(resume_path) else None
	if resume_state is None and CFG.resume and CFG.resume_from and os.path.exists(CFG.resume_from):
		resume_state = load_resume_state(CFG.resume_from)

	start_epoch = 0
	start_batch = 0
	stage = 1
	best_f1 = 0.0
	best_ep = 0
	no_imp = 0
	history = default_history()
	train_accum = {'loss_sum': 0.0, 'seen': 0, 'preds': [], 'labels': []}

	if resume_state is not None:
		print(f'\n*** RESUMING from checkpoint ***', flush=True)
		start_epoch = resume_state['epoch']
		start_batch = resume_state['batch_idx']
		stage = resume_state['stage']
		best_f1 = resume_state['best_f1']
		best_ep = resume_state['best_ep']
		no_imp = resume_state['no_imp']
		history = normalize_history(resume_state['history'])
		print(f'  epoch={start_epoch} batch={start_batch} stage={stage} best_f1={best_f1:.4f} best_ep={best_ep}', flush=True)

	if start_batch > 0:
		train_accum = resume_state['train_accum'] or train_accum
		exact_keys = (
			resume_state is not None and
			resume_state.get('optimizer_state') is not None and
			resume_state.get('scheduler_state') is not None and
			resume_state.get('ema_state') is not None and
			resume_state.get('rng_state') is not None and
			resume_state.get('train_accum') is not None
		)
		if not exact_keys:
			print('  WARNING: checkpoint is incomplete for mid-epoch resume; restarting this epoch from batch 0.', flush=True)
			start_batch = 0
			train_accum = {'loss_sum': 0.0, 'seen': 0, 'preds': [], 'labels': []}
	else:
		train_accum = {'loss_sum': 0.0, 'seen': 0, 'preds': [], 'labels': []}

	if stage == 1:
		set_stage1_trainability(model)
		optimizer = build_stage1_optimizer(model)
		scheduler = build_stage1_scheduler(optimizer)
	else:
		set_stage2_trainability(model)
		optimizer = build_stage2_optimizer(model)
		scheduler = build_stage2_scheduler(optimizer)

	ema = CPUBackedEMA(model, CFG.ema_decay)
	swa = CPUBackedSWA()
	scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE_TYPE == 'cuda' and AMP_DTYPE == torch.float16))

	if resume_state is not None:
		load_trainable_state(model, resume_state['trainable_state'])
		if resume_state.get('optimizer_state') is not None:
			try:
				optimizer.load_state_dict(resume_state['optimizer_state'])
				# Upcast bf16 momentum/variance buffers back to float32
				# Without this, in-place ops keep them bf16 forever, degrading optimization
				_upcast_count = 0
				for _state in optimizer.state.values():
					for _k, _v in _state.items():
						if isinstance(_v, torch.Tensor) and _v.dtype == torch.bfloat16:
							_state[_k] = _v.float()
							_upcast_count += 1
				print(f'  Loaded optimizer state (momentum/variance preserved, {_upcast_count} tensors upcast to fp32)', flush=True)
			except Exception as e:
				print(f'  WARNING: Failed to load optimizer state: {e}', flush=True)
		else:
			print(f'  WARNING: No optimizer state in checkpoint (lightweight save). Momentum will rebuild.', flush=True)
		if resume_state.get('scheduler_state') is not None and scheduler is not None:
			try:
				scheduler.load_state_dict(resume_state['scheduler_state'])
			except Exception:
				pass
		if resume_state.get('ema_state') is not None:
			ema.load_state_dict(resume_state['ema_state'])
		if resume_state.get('scaler_state') is not None and scaler.is_enabled():
			try:
				scaler.load_state_dict(resume_state['scaler_state'])
			except Exception:
				pass
		ema.extend(model)
		# Restore SWA state if it was saved
		if resume_state.get('swa_state') is not None:
			swa.load_state_dict(resume_state['swa_state'], resume_state.get('swa_n', 0))
			print(f'  Restored SWA state ({swa.n} snapshots)', flush=True)
		restore_rng_state(resume_state.get('rng_state'))
		del resume_state
		gc.collect()
	else:
		ema.extend(model)

	tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
	print(f'\nStage {stage} trainable params: {tp / 1e9:.2f}B', flush=True)
	if torch.cuda.is_available():
		print(f'  VRAM after setup: {torch.cuda.memory_allocated() / 1e9:.1f} GB', flush=True)

	for epoch in range(start_epoch, CFG.epochs):
		if time.time() > deadline:
			print('\n[TIME LIMIT] saving final checkpoint and stopping before the session ends.', flush=True)
			save_resume_checkpoint(
				resume_path, model, optimizer, scheduler, ema, swa, scaler,
				epoch, 0, stage, CFG.seed + epoch * 997 + stage * 101,
				best_f1, best_ep, no_imp, history, train_accum,
				lightweight=False,
			)
			save_dashboard(history, best_f1, best_ep, work_dir, CFG.output_prefix)
			break

		if epoch == CFG.lp_epochs and stage == 1:
			stage = 2
			set_stage2_trainability(model)
			optimizer = build_stage2_optimizer(model)
			scheduler = build_stage2_scheduler(optimizer)
			ema.extend(model)
			tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
			print(f'\n--- Stage 2: unfreezing last {CFG.unfreeze_last_n} blocks ({tp/1e9:.2f}B trainable) ---', flush=True)

		epoch_seed = CFG.seed + epoch * 997 + stage * 101
		loader = make_train_loader(tr_ds, df_tr, epoch_seed)
		print(f'\n[Epoch {epoch + 1}/{CFG.epochs}] stage={stage} seed={epoch_seed} start_batch={start_batch}', flush=True)

		train_stats = train_one_epoch(
			model, loader, optimizer, criterion, ema, swa, scaler, scheduler,
			epoch, stage, epoch_seed, start_batch, deadline, resume_path,
			history, best_f1, best_ep, no_imp, train_accum,
		)

		if train_stats['interrupted']:
			break

		start_batch = 0
		train_accum = train_stats['train_accum']
		scheduler.step()

		# SWA: accumulate snapshot after each FT epoch past the start point
		if epoch >= CFG.swa_start_epoch and stage == 2:
			swa.update(model)
			print(f'  [SWA] snapshot #{swa.n} accumulated (epoch {epoch+1})', flush=True)

		val_stats = evaluate(model, ema, va_loader, criterion)

		lr_now = optimizer.param_groups[-1]['lr']
		history['tl'].append(float(train_stats['train_loss']))
		history['vl'].append(float(val_stats['val_loss']))
		history['tf'].append(float(train_stats['train_f1']))
		history['vf'].append(float(val_stats['raw_f1']))
		history['ef'].append(float(val_stats['ema_f1']))
		history['vp'].append([float(x) for x in val_stats['best_pc']])
		history['lr'].append(float(lr_now))

		improved = val_stats['best_f1'] > best_f1
		if improved:
			best_f1 = val_stats['best_f1']
			best_ep = epoch + 1
			no_imp = 0
			save_best_checkpoint(best_path, model, ema, val_stats['use_ema'], stage, best_f1, best_ep)
		else:
			no_imp += 1

		# Epoch-end save: FULL checkpoint with optimizer state (compressed to bf16)
		try:
			save_resume_checkpoint(
				resume_path, model, optimizer, scheduler, ema, swa, scaler,
				epoch + 1, 0, stage, epoch_seed,
				best_f1, best_ep, no_imp, history, train_accum,
				lightweight=False,
			)
		except OSError:
			print('  Full epoch save failed (disk), saving lightweight...', flush=True)
			save_resume_checkpoint(
				resume_path, model, optimizer, scheduler, ema, swa, scaler,
				epoch + 1, 0, stage, epoch_seed,
				best_f1, best_ep, no_imp, history, train_accum,
				lightweight=True,
			)

		with open(history_path, 'w') as f:
			json.dump(history, f, indent=2)

		tag = 'EMA' if val_stats['use_ema'] else 'raw'
		star = ' *** BEST' if improved else f' ({no_imp}/{CFG.early_stopping_patience})'
		peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
		print(
			f'Ep {epoch + 1:02d} | LR {lr_now:.1e} | '
			f'TrL {train_stats["train_loss"]:.4f} TrF1 {train_stats["train_f1"]:.4f} | '
			f'VaL {val_stats["val_loss"]:.4f} VaF1 {val_stats["raw_f1"]:.4f} EmaF1 {val_stats["ema_f1"]:.4f} | '
			f'Best {best_f1:.4f} ({tag}){star} | Peak {peak:.0f}GB',
			flush=True,
		)
		print('  ' + ' | '.join(f'{CLASS_NAMES[i]}:{val_stats["best_pc"][i]:.3f}' for i in range(CFG.num_classes)), flush=True)

		if no_imp >= CFG.early_stopping_patience and stage == 2:
			print(f'\n[EARLY STOP] Best F1 {best_f1:.4f} at epoch {best_ep}', flush=True)
			break

	# ---- Save dashboard + SWA model at the very end ----
	save_dashboard(history, best_f1, best_ep, work_dir, CFG.output_prefix)

	if swa.is_active():
		swa_path = os.path.join(work_dir, f'{CFG.output_prefix}_swa.pth')
		print(f'\n[SWA] Saving SWA model ({swa.n} snapshots averaged, epochs {CFG.swa_start_epoch+1}..{epoch+1})', flush=True)
		save_checkpoint(swa_path, build_swa_payload(swa, stage))
	else:
		print(f'\n[SWA] No snapshots accumulated (training ended before epoch {CFG.swa_start_epoch+1})', flush=True)

	print(f'\nDone. Best F1: {best_f1:.4f} at epoch {best_ep}', flush=True)
	print(f'Resume checkpoint: {resume_path}', flush=True)


if __name__ == '__main__':
	main()


## Run TrainingExecute `main()` to start the full LP-FT pipeline.